# LLMSlim round-3 benchmark builds: `lme2`

This notebook builds the LongMemEval-S baselines and competitors (180 test questions, histories of about 115k tokens): the full histories, the most recent turns, RAG over turns (Qwen3-Embedding-0.6B) and LLMLingua-2, at 2,500 and 10,000 tokens. It replaces the `lme` notebook, which ran out of GPU memory embedding long turns. LLMSlim's LongMemEval arms are built by the `llmslim_v23` notebook.

It **never calls a model API**. It only produces the exact context each system would send to the reader, so anyone can rerun it and compare the outputs byte for byte. The reader (Claude Haiku 5.5) and the judge run on these published contexts afterwards.

- **Code:** [Thanatos9404/llmslim](https://github.com/Thanatos9404/llmslim) at commit `e93e844b0cf11af2bd066daa8f79860166f69788`, `benchmarks/live/round3/`. Splits were committed before any result: `splits_r3.json`.
- **Data:** downloaded from each dataset's original source (lost-in-the-middle, HotpotQA, LoCoMo, LongMemEval) and checked by sha256.
- **Environments:** pinned Python 3.12 venvs (`kaggle_run.py setup`), separate from Kaggle's image.
- **Output:** `round3_lme2.zip`: one JSONL per system and suite (`id`, `context`, `tokens`, `build_ms`) and `manifest-lme2.json` (commit, GPUs, package versions, data and output hashes).

Settings: accelerator **GPU T4 x2**, internet **on**. Runtime: about 7 hours (LLMLingua-2 reads every 115k-token history once per budget). A rerun resumes from the cases already built.

In [ ]:
import subprocess
import sys

COMMIT = "e93e844b0cf11af2bd066daa8f79860166f69788"
JOB = "lme2"
R = "/tmp/src/benchmarks/live/round3/kaggle_run.py"
subprocess.run(
    "rm -rf /tmp/src && git init -q /tmp/src && cd /tmp/src && git remote add origin https://github.com/Thanatos9404/llmslim && git fetch -q --depth 1 origin "
    + COMMIT
    + " && git checkout -q FETCH_HEAD && git log -1 --format='%H %s'",
    shell=True,
    check=True,
)
subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv", shell=True, check=True
)
subprocess.run([sys.executable, R, "setup", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "data", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "build", "--work", "/tmp/r3", "--job", JOB], check=True)
subprocess.run(
    [
        sys.executable,
        R,
        "package",
        "--work",
        "/tmp/r3",
        "--job",
        JOB,
        "--zip",
        "/kaggle/working/round3_" + JOB + ".zip",
    ],
    check=True,
)